In [3]:
from collections import Counter
from pathlib import Path
import pickle
import automated_llm_probes as alp

TARGET_N = 300
NAMES = ['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

HUMAN_N = {"brick": 2019, "knife": 1028, "car tires": 960, "box": 833, "rope": 829,
    "pen": 742, "wooden slat": 671, "paperclip": 534, "tin can": 425,
    "socks": 339, "light bulb": 337, "spoon": 337, "towel": 327, "book": 326,
    "belt": 300, "bucket": 300, "sock": 300, "candle": 299}

def targets(n):
    tot = sum(HUMAN_N.values())
    raw = {c: n * k / tot for c, k in HUMAN_N.items()}
    out = {c: int(v) for c, v in raw.items()}
    for c in sorted(raw, key=lambda c: raw[c] - out[c], reverse=True):
        if sum(out.values()) >= n:
            break
        out[c] += 1
    return out

tgt = targets(TARGET_N)
models = [m for m in alp.ready_models() if m["name"] in NAMES]
print("N", TARGET_N, [m["name"] for m in models])

for m in models:
    have = Counter()
    for p in Path("aut", m["name"]).rglob("*.pickle"):
        try:
            row = pickle.load(open(p, "rb"))
        except Exception:
            continue
        cue = (row.get("kwargs") or {}).get("cue") or row.get("cue")
        if cue:
            have[str(cue).strip().lower()] += 1
    n_have = sum(have.values())
    print(f"\n{m['name']}  {n_have}")
    for cue, want in tgt.items():
        need = max(0, want - have.get(cue, 0))
        print(f"  {cue:16s} {have.get(cue, 0):4d}/{want:<3d}  {'ok' if need == 0 else f'+{need}'}")
        if need:
            alp.collect("AUT", models=[m], n_per_model=n_have + need, cue=cue)
            n_have += need

SyntaxError: invalid syntax (3883878317.py, line 4)